# OP-02 · Référence observée CHIRPS : archive et normales

**Notebook opérationnel.**

| | |
|---|---|
| Étape du workflow | E1 — observations de référence |
| Entrée | CHIRPS v2.0 quotidien (`observations.precip.daily_path` du fichier de cycle) |
| Sorties | `DATA_OSF/derived/obs/chirps/` : cumuls décadaires et mensuels (0,05° et 1°), normales 1991–2020, contrôle qualité |
| Quand l'exécuter | quand le fichier CHIRPS quotidien est mis à jour. Sinon l'étape détecte que la source n'a pas changé et ne recalcule rien. |
| Durée | environ 10 min pour une reconstruction complète ; quelques secondes sinon |

Équivalent en ligne de commande : `python scripts/run_obs_chirps.py --config config/cycle_YYYYMM.yaml`

## Paramètres

In [ ]:
CYCLE_CONFIG   = "config/cycle_202611.yaml"
CHIRPS_ANNEES  = None      # None = l'année du dernier mois écoulé (complétée à chaque cycle) ;
                           # ou (1981, 2026) pour (re)constituer l'archive
CHIRPS_REFRESH = False     # True : retélécharger ces années même si elles semblent à jour
REBUILD        = False     # True : tout recalculer même si la source CHIRPS est inchangée

In [ ]:
from pathlib import Path
import os
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)
from eccas_s2s.settings import load_cycle
cfg = load_cycle(CYCLE_CONFIG)
print(f"Cycle {cfg.cycle_id} — initialisation {cfg.init_date.date()}")

## 0. Mise à jour de CHIRPS (année en cours)

À chaque cycle, l'année en cours doit recevoir le mois qui vient de s'écouler. Le fichier local
d'une année incomplète (qui s'arrête avant le 31 décembre) est comparé au fichier publié par
UCSB (date `Last-Modified`, lue par une requête légère) : s'il en existe un plus récent, l'année
est retéléchargée, découpée sur la CEEAC et remplace l'ancienne. Une année complète n'est
jamais retéléchargée. Plus besoin de supprimer le fichier à la main.

L'étape suivante recalcule les cumuls et les normales dès que le fichier source a changé.

In [ ]:
from eccas_s2s.operations import download_chirps

first, last = CHIRPS_ANNEES or (None, None)
dl = download_chirps.run(CYCLE_CONFIG, first, last, refresh=CHIRPS_REFRESH)
print(f"\nStatut : {dl.status} — années mises à jour : {dl.parameters.get('updated_years') or 'aucune'}")

## 1. Archive, passage à 1° et normales

In [ ]:
from eccas_s2s.operations import obs_chirps
ctx = obs_chirps.run(CYCLE_CONFIG, rebuild=REBUILD)
print(f"\nStatut : {ctx.status}")
for w in ctx.warnings:
    print(" ⚠", w)

## 2. Contrôle qualité

In [ ]:
paths = obs_chirps.derived_paths(cfg)
qc = pd.read_csv(paths["qc"])
print(f"{len(qc)} mois contrôlés ({qc.year.min()}–{qc.year.max()})")
pd.Series({
    "mois incomplets": int((qc.days_present != qc.days_expected).sum()),
    "valeurs manquantes sur le domaine": int(qc.missing_land_values.sum()),
    "valeurs négatives": int(qc.negative_values.sum()),
    "valeurs > 300 mm/j": int(qc.suspicious_values.sum()),
}).to_frame("bilan")

## 3. Fichiers produits

In [ ]:
pd.DataFrame([{"fichier": p.name, "taille (Mo)": round(p.stat().st_size / 1e6, 1)}
              for k, p in paths.items() if k != "dir"])

In [ ]:
print("manifeste :", ctx.run_dir / "manifest.json")